# 03 — Data Sampling: From Token IDs to Training Batches

**Lecture goal:** learn what a PyTorch *tensor* is and why we need one, then build the pipeline that turns one long sequence of token IDs into the (input, target) batches a model actually trains on.

## The training objective, in one sentence

A GPT-style model is trained to do exactly one thing: **given some tokens, predict the next token.** That's it — that single, simple objective, applied at massive scale, is what produces everything from grammar to reasoning-like behavior. So every training example is a pair:

```
input:  the tokens the model gets to see
target: the tokens it should have predicted (input, shifted one position to the right)
```

Before we can build that, we need our first piece of PyTorch: **tensors**.

## What is a tensor, and why not just use Python lists?

A **tensor** is PyTorch's core data structure. At its simplest, it's just a grid of numbers — but generalized to any number of dimensions:

- A single number (`5`) is a **0-dimensional** tensor (a "scalar").
- A list of numbers (`[1, 2, 3]`) is a **1-dimensional** tensor (a "vector").
- A grid of numbers (rows and columns) is a **2-dimensional** tensor (a "matrix").
- Stack matrices together and you get a **3-dimensional** tensor, and so on.

We *could* represent all of this with nested Python lists. So why doesn't PyTorch just use those? Two reasons that matter enormously once models get big:

1. **Speed.** Tensor operations (adding, multiplying, reshaping millions of numbers) run as compiled, vectorized code — often on a GPU — instead of a slow Python `for` loop over each element.
2. **Automatic differentiation.** PyTorch can automatically track every operation performed on a tensor and compute *gradients* through it — the derivatives that tell us how to adjust the model's parameters during training. We'll see this in action in notebook 10; for now, just know it's why tensors exist instead of plain lists.

Let's create a few tensors and get comfortable with the basics: shape, dtype, and indexing.

In [1]:
import torch

print("PyTorch version:", torch.__version__)

# A 1-D tensor, built directly from a Python list.
vector = torch.tensor([10, 20, 30, 40])
print(vector)
print("shape:", vector.shape)   # how many elements along each dimension
print("dtype:", vector.dtype)   # the numeric type stored in the tensor

PyTorch version: 2.13.0+cpu
tensor([10, 20, 30, 40])
shape: torch.Size([4])
dtype: torch.int64


In [2]:
# A 2-D tensor: think of it as a table with rows and columns.
matrix = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
])
print(matrix)
print("shape:", matrix.shape)  # (rows, columns) = (2, 3)

tensor([[1, 2, 3],
        [4, 5, 6]])
shape: torch.Size([2, 3])


`shape` is the single most important attribute you'll check when debugging PyTorch code — almost every bug in a neural network shows up first as "the shapes don't match." Get comfortable reading it: `torch.Size([2, 3])` means 2 rows, 3 columns; a shape of `[8, 4, 256]` (which we'll see very soon) means "8 things, each with 4 things, each of those made of 256 numbers."

Indexing and slicing works just like Python lists / NumPy arrays:

In [3]:
print("First row:        ", matrix[0])
print("Element (row 1, col 2):", matrix[1, 2])
print("First two columns:\n", matrix[:, :2])

First row:         tensor([1, 2, 3])
Element (row 1, col 2): tensor(6)
First two columns:
 tensor([[1, 2],
        [4, 5]])


That's enough tensor background to get moving — we'll introduce more operations (matrix multiplication, broadcasting, etc.) exactly when we first need them, in later notebooks.

## Step 1: Encode the full text into one long sequence of token IDs

We'll reuse the BPE tokenizer from notebook 02.

In [4]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

encoded_text = tokenizer.encode(raw_text)
print("Total number of tokens:", len(encoded_text))

Total number of tokens: 5145


## Step 2: Sliding a fixed-size window over the token sequence

We pick a **context length** (also called block size or sequence length) — how many tokens of "history" the model gets to look at when predicting the next one. Real GPT-2 uses 1024; we'll use a small number like 4 here so the examples are easy to read, and scale up later.

For each starting position, the **input** is a chunk of `context_length` consecutive tokens, and the **target** is the same chunk shifted one position to the right — i.e. "the next token after each input token."

In [5]:
context_length = 4

x = encoded_text[0:context_length]
y = encoded_text[1:context_length + 1]

print("Input IDs: ", x)
print("Target IDs:", y)

Input IDs:  [40, 367, 2885, 1464]
Target IDs: [367, 2885, 1464, 1807]


Let's make the "predict the next token" objective fully concrete by printing every partial input inside this one window, alongside what it should predict next:

In [6]:
for i in range(1, context_length + 1):
    context = encoded_text[:i]
    target = encoded_text[i]
    print(f"{context} ---> {target}")
    print(f"{tokenizer.decode(context)!r} ---> {tokenizer.decode([target])!r}")

[40] ---> 367
'I' ---> ' H'
[40, 367] ---> 2885
'I H' ---> 'AD'
[40, 367, 2885] ---> 1464
'I HAD' ---> ' always'
[40, 367, 2885, 1464] ---> 1807
'I HAD always' ---> ' thought'


That's the entire supervised-learning signal a GPT model is trained on, laid bare: given a prefix of tokens, predict the single token that comes next. Every one of these `(context, target)` pairs, at every position, in every document, across the whole training corpus, is one training example.

## Step 3: Sliding the window across the *whole* sequence, with a stride

We don't stop at one window — we slide it across the entire token sequence to generate many training examples. The **stride** controls how far we move the window each step:

- `stride == context_length`: windows don't overlap at all — maximizes how much distinct text we cover per pass over the data.
- `stride == 1`: windows overlap almost completely, each shifted by a single token — maximizes the *number* of training examples we can extract from limited text, at the cost of examples being highly correlated with their neighbors.

Let's implement this with plain Python first, to see the mechanism, before wrapping it in PyTorch's data-loading tools.

In [7]:
def sliding_window_pairs(token_ids, context_length, stride):
    inputs, targets = [], []
    for start in range(0, len(token_ids) - context_length, stride):
        inputs.append(token_ids[start : start + context_length])
        targets.append(token_ids[start + 1 : start + context_length + 1])
    return inputs, targets


inputs, targets = sliding_window_pairs(encoded_text, context_length=4, stride=4)
print("Number of (input, target) pairs:", len(inputs))
for i in range(3):
    print("input: ", inputs[i])
    print("target:", targets[i])
    print()

Number of (input, target) pairs: 1286
input:  [40, 367, 2885, 1464]
target: [367, 2885, 1464, 1807]

input:  [1807, 3619, 402, 271]
target: [3619, 402, 271, 10899]

input:  [10899, 2138, 257, 7026]
target: [2138, 257, 7026, 15632]



## Step 4: PyTorch `Dataset` and `DataLoader`

We *could* keep using plain Python lists and loops like above. But once we start training, we need to:

- group examples into **batches** (feed the model several examples at once — GPUs process a batch in parallel, close to as fast as processing one example alone),
- **shuffle** examples each epoch (so the model doesn't learn spurious patterns from the order the text happens to appear in),
- do this efficiently without loading the entire dataset into memory in Python-list form.

PyTorch provides two classes for exactly this:

- **`Dataset`**: you tell it how many examples exist (`__len__`) and how to fetch the *i*-th one (`__getitem__`). It doesn't do any batching itself — it's just a uniform way to say "here is example number `i`."
- **`DataLoader`**: wraps a `Dataset` and handles batching, shuffling, and iteration for you. You ask it for a batch, it hands you a stack of examples as tensors.

Let's write our own `Dataset` subclass that does the sliding-window logic we just wrote by hand, but returns PyTorch tensors.

In [8]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        # Same sliding-window logic as sliding_window_pairs above, but storing tensors.
        for start in range(0, len(token_ids) - context_length, stride):
            input_chunk = token_ids[start : start + context_length]
            target_chunk = token_ids[start + 1 : start + context_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # DataLoader calls this to know how many examples exist in total.
        return len(self.input_ids)

    def __getitem__(self, idx):
        # DataLoader calls this to fetch one example at a time (before batching them together).
        return self.input_ids[idx], self.target_ids[idx]

In [9]:
def create_dataloader_v1(
    text, batch_size=4, context_length=256, stride=128, shuffle=True, drop_last=True
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, context_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,  # drop the final batch if it would be smaller than batch_size
    )
    return dataloader

Let's build a small dataloader and pull out the first batch to see what shape comes out the other end.

In [10]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, context_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs shape: ", inputs.shape)
print(inputs)
print()
print("Targets shape:", targets.shape)
print(targets)

Inputs shape:  torch.Size([8, 4])
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets shape: torch.Size([8, 4])
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


Shape `[8, 4]` means: a batch of 8 examples, each 4 tokens long — exactly `(batch_size, context_length)`. This is the shape every downstream piece of the model (embeddings, attention, everything) will expect as its input. Whenever something breaks later, the first thing to check is whether a tensor's shape matches what the next step expects.

Let's also confirm `stride < context_length` produces overlapping windows, as promised:

In [11]:
overlap_loader = create_dataloader_v1(
    raw_text, batch_size=2, context_length=4, stride=1, shuffle=False
)
inputs, targets = next(iter(overlap_loader))
print("Inputs (stride=1, notice each row shifts by only one token):")
print(inputs)

Inputs (stride=1, notice each row shifts by only one token):
tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]])


## Recap

- A **tensor** is PyTorch's grid-of-numbers data structure — like nested Python lists, but fast (vectorized, GPU-capable) and capable of automatic differentiation.
- A GPT model's entire training signal is: given a chunk of tokens, predict the next one. We generate these `(input, target)` pairs with a **sliding window** over the token sequence.
- **`stride`** controls overlap between windows: smaller stride = more (and more correlated) examples from the same text.
- PyTorch's **`Dataset`** (defines how to fetch one example) and **`DataLoader`** (batches, shuffles, iterates) turn this into an efficient, reusable pipeline that hands us tensors of shape `(batch_size, context_length)`.

### What's next

We now get batches of *integer* token IDs. But a neural network's inner workings are entirely built from continuous, real-valued numbers it can do arithmetic and calculus on — an integer ID like `40` carries no useful notion of "how similar is this token to that one." In notebook 04, we'll convert every token ID into a vector of numbers — an **embedding** — that the model can actually learn from, and add positional information so the model knows *where* in the sequence each token sits.